In [2]:
import pandas as pd
import numpy as np

In [3]:
!pip install pymongo[srv] requests pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 40.4 MB/s eta 0:00:00


In [4]:
# ==========================================
# PART 3: EXTRACT (Dune) -> LOAD (MongoDB)
# ==========================================

import requests                 # Library to talk to the Dune API
from pymongo import MongoClient # Library to talk to MongoDB

# ==========================================
# CONFIGURATION (FILL THESE IN CAREFULLY!)
# ==========================================

# 1. Your Dune API Key
# Paste the long key you just copied from the Dune settings popup.
DUNE_API_KEY = "Your Dune API Key"

# 2. Your Dune Query ID
# This is the number from your Dune URL (e.g., dune.com/queries/1234567 -> 1234567)
QUERY_ID = "Dune query id"

# 3. Your MongoDB Connection String
# Go to Atlas -> Database -> Connect -> Drivers -> Python -> Copy the string.
# It usually starts with "mongodb+srv://" and has your username/password.
MONGO_URI = "your driver string"

# ==========================================
# FUNCTION 1: GET DATA FROM DUNE
# ==========================================
def fetch_dune_data():
    print("Step 1: Contacting Dune Analytics...")

    # The API URL format requires the Query ID
    url = f"https://api.dune.com/api/v1/query/{QUER_ID}/results"

    # We pass the API Key in the header (like an ID card)
    headers = {"X-Dune-Api-Key": DUNE_API_KEY}

    # Send the request
    response = requests.get(url, headers=headers)

    # Check if the request succeeded (Status 200 means OK)
    if response.status_code == 200:
        data = response.json()
        # The actual data rows are nested inside result -> rows
        rows = data['result']['rows']
        print(f" -> Success! Retrieved {len(rows)} rows from Dune.")
        return rows
    else:
        print(f" -> Error: {response.text}")
        return None

# ==========================================
# FUNCTION 2: UPLOAD TO MONGODB
# ==========================================
def upload_to_mongo(data):
    if not data:
        print("No data to upload. Stopping.")
        return

    print("Step 2: Connecting to MongoDB...")

    try:
        # Connect to your Atlas cluster
        client = MongoClient(MONGO_URI)

        # Access  the database 'Stablecoin_Project'
        db = client['Stablecoin_Project']

        # Accessing  the collection 'Q3_2024'
        # (To matche match my assigned quarter)
        collection = db['Q3_2024']

        # Insert the data
        print(f" -> Uploading {len(data)} documents...")
        collection.insert_many(data)
        print(" -> Done! Data successfully saved to MongoDB.")

    except Exception as e:
        print(f" -> MongoDB Error: {e}")

# ==========================================
# MAIN EXECUTION
# ==========================================

# 1. Fetch
dune_data = fetch_dune_data()

# 2. Upload
upload_to_mongo(dune_data)

Step 1: Contacting Dune Analytics...
 -> Success! Retrieved 368 rows from Dune.
Step 2: Connecting to MongoDB...
 -> Uploading 368 documents...
 -> Done! Data successfully saved to MongoDB.
